# 61. 热力图（heatmap）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 18 / 20 步：组织多变量、矩阵和分面证据**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 成对关系图（pairplot）  →  **本章任务：** 热力图（heatmap）  →  **下一步：** 聚类热力图（clustermap）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

热力图用颜色深浅替你把一整张数值表读出来，几十个甚至上百个格子扫一眼就能看出哪些组合数值偏高、哪些偏低、哪里是空白区。



## 本章目标

学完本章，你将能够：

- **理解**：理解「热力图（heatmap）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「热力图（heatmap）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「热力图（heatmap）」并读出其中的结论。


## 61.1 适用场景

**背景引入**：热力图用颜色深浅替你把一整张数值表读出来，几十个甚至上百个格子扫一眼就能看出哪些组合数值偏高、哪些偏低、哪里是空白区。做相关分析、客单价对比这类“行列交叉”的数据时，散点图和条形图都装不下这么多组合，热力图就成了最直观的那张图。

打个比方：heatmap 像'把一张满是数字的表格涂成温度地图'——数字越大颜色越深，扫一眼就能看到哪些格是'热点'、哪些是'冷区'、哪里整块偏亮或偏暗，省得你逐格读几十上百个格子。它最擅长回答'哪两列交叉处数值偏高/偏低'这类行列交叉的问题。

比较行×列组合或压缩读取数值矩阵。


## 61.2 数据结构

二维矩阵或可透视成长×宽矩阵的长表。


## 61.3 本章练习任务

运行基础图表后，完成以下任务：

1. 将 cmap="vlag" 改为 cmap="coolwarm" 或 "RdBu_r"，对比不同发散色盘的视觉效果
2. 修改 center=0 为不设置 center，观察色阶中心对相关矩阵显示的影响
3. 调整 fmt=".2f" 为 fmt=".0f"，说明标注精度对数值可读性的作用


## 61.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `np.triu()`、`np.ones_like()`、`plt.subplots()`、`sns.heatmap()` | 比较行×列组合或压缩读取数值矩阵。 | 色阶范围随数据变化无法跨图比较 |
| 进阶变体 | `orders.pivot_table()`、`plt.subplots()`、`sns.heatmap()`、`ax.set()` | 在基础图表上增加分组、注释、布局或交互 | 相关矩阵使用单向色盘 |
| 关键参数 | `annot` | 标注 | 色阶范围随数据变化无法跨图比较 |
| 关键参数 | `fmt` | 格式 | 相关矩阵使用单向色盘 |
| 关键参数 | `cmap` | 色盘 | 标注过密 |
| 关键参数 | `center/vmin/vmax` | 色阶 | 色阶范围随数据变化无法跨图比较 |


## 61.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-61 -->
### 数学推导｜矩阵颜色必须对应明确的数值变换

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜按列估计中心与尺度。** $\mu_j=\frac1n\sum_ix_{ij}$，$\sigma_j^2=\frac1{n-1}\sum_i(x_{ij}-\mu_j)^2$。

**第 2 步｜把原值换成离均值多少个标准差。** $z_{ij}=(x_{ij}-\mu_j)/\sigma_j$。

**第 3 步｜检查变换结果。** 对非零方差列，标准化后近似满足

$$
\frac1n\sum_i z_{ij}\approx0,
\qquad
\frac1{n-1}\sum_i z_{ij}^2=1
$$

因此不同原始单位可以共享色阶，但颜色不再表示原单位。

**把上面的关系收束为本章计算式：**

$$
z_{ij}=\frac{x_{ij}-\mu_j}{\sigma_j}
$$

**符号解释：** $z_{ij}$ 是按列标准化后的值，使不同单位的列可在同一色阶比较。

**代码对应：** 根据问题选择原值、比例、相关系数或 z-score，再设置统一色阶。

**使用边界：** 标准化会丢失原单位；色阶中心、范围和缺失值颜色都必须说明。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 61.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

corr = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("营销指标相关系数")
fig.tight_layout()
plt.show()


**练一练**：基础图表把 `visits`、`ad_spend`、`sales`、`conversion` 四个指标一起画进相关矩阵。现在只看「`sales` 与其他指标」的一条信息：把相关系数矩阵精简到只保留 `sales` 这一行，再画热力图，看看哪些指标和销售额关系更近。

下面给出脚手架，把 `sales_corr` 填成正确的 DataFrame（只含 `sales` 一行的相关系数)后会通过自检；完成版见右侧「参考答案」。


In [ ]:
# 请在下方填写代码
# 目标：把 marketing 相关矩阵精简成「只看 sales 这一行」的 DataFrame，保存为 sales_corr
# 提示：
#   1. corr_df = marketing[["visits","ad_spend","sales","conversion"]].corr()
#   2. 用 .loc[["sales"]] 只取 sales 这一行，赋给 sales_corr
#   3. 可以再用 sns.heatmap 把 sales_corr 画出来
# 把下面的 你的代码 替换成你的实现：
corr_df = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
sales_corr = 你的代码  # 例如 corr_df.loc[["sales"]]

fig, ax = plt.subplots(figsize=(6, 2.5))
sns.heatmap(
    sales_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("sales 与各指标的相关系数")
fig.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 完整答案：只保留 sales 一行相关系数并画热力图
corr_df = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
sales_corr = corr_df.loc[["sales"]].copy()  # 只看 sales 这一行，共 4 列
fig, ax = plt.subplots(figsize=(6, 2.5))
sns.heatmap(
    sales_corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("sales 与各指标的相关系数")
fig.tight_layout()
plt.show()


## 61.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

pivot = orders.pivot_table(
    index="region", columns="category", values="order_value", aggfunc="mean"
)
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".0f",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={"label": "平均客单价（元）"},
    ax=ax,
)
ax.set(title="区域与品类客单价", xlabel="品类", ylabel="区域")
fig.tight_layout()
plt.show()


## 61.8 参数说明

- annot：标注
- fmt：格式
- cmap：色盘
- center/vmin/vmax：色阶


## 61.9 结果解读

先读色阶含义，再找极值、带状结构和异常组合。


## 61.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 61.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 61.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 61.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 61.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 61.12 易错点提醒

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


## 61.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 61.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：只保留 sales 与各指标的相关关系，聚焦单变量
# 【目标】用 .loc 只取一行，聚焦「销售额和谁最相关」这一个问题。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：取 sales 这一行，画成一张单行热力图。
corr = marketing[["visits", "ad_spend", "sales", "conversion"]].corr()
sales_corr = corr.loc[["sales"]]
fig, ax = plt.subplots(figsize=(7, 2.2))
sns.heatmap(sales_corr, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("sales 与各指标相关系数")
fig.tight_layout()
plt.show()

# ---- 反思记录：与 sales 最相关/最不相关的指标是谁 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

counts = pd.crosstab(orders["region"], orders["channel"])
fig, ax = plt.subplots(figsize=(7.5, 4))
sns.heatmap(counts, annot=True, fmt="d", cmap="YlGnBu", linewidths=0.5, ax=ax)
ax.set(title="区域与渠道订单量", xlabel="渠道", ylabel="区域")
fig.tight_layout()
plt.show()


## 61.15 小结

用颜色矩阵展示相关系数、透视表或任意二维数值。


### 61.15.1 你已经掌握

- 判断热力图（heatmap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 61.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `annot` | 标注 |
| `fmt` | 格式 |
| `cmap` | 色盘 |
| `center/vmin/vmax` | 色阶 |


### 61.15.3 需要注意

- 色阶范围随数据变化无法跨图比较
- 相关矩阵使用单向色盘
- 标注过密


### 61.15.4 完成检查

- [ ] 能判断什么问题适合使用热力图（heatmap）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 61.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
